# Model 1 baseline on Kaggle

Use the Kaggle **GPU T4 x2** accelerator. Enable Internet for the initial clone and attach the G1 mesh dataset for training. Run the cells from the top.

M1 trains for 1.6 million environment steps with 8 environments (4 per T4) and seed 42. Its actor uses the current observation. The notebook verifies the checkpoint and reports M1 on ten matched seeds before and after training. You can attach a saved M3 run later for the optional comparison.

In [ ]:
import json
import os
from pathlib import Path
import shutil
import statistics
import subprocess
import sys
from zipfile import ZipFile

repo = Path("/kaggle/working/Bipedal-Locomotion")
inputs = Path("/kaggle/input")
branch = "m3-kaggle"
url = "https://github.com/TotallyNotMinh/Bipedal-Locomotion.git"

if not repo.exists():
    subprocess.run(["git", "clone", "--branch", branch, "--single-branch", url, str(repo)], check=True)
active_branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=repo, text=True).strip()
if active_branch != branch:
    raise RuntimeError(f"Expected {branch} in {repo}, found {active_branch!r}. Check the existing checkout.")
subprocess.run(["git", "log", "-1", "--oneline"], cwd=repo, check=True)

run = repo / "results/m1_1p6m_env8_lr1e4"
budget = 1_600_000
num_envs = 8
seed = 42
env = {**os.environ, "XLA_PYTHON_CLIENT_PREALLOCATE": "false"}

def read(folder, filename):
    return json.loads((folder / filename).read_text())

## Load the G1 meshes for training

For training, the notebook copies missing STL files from `/kaggle/input/datasets/trung1311/robot-assets/results/m3_robot_assets` into the trainer’s mesh directory. Checkpoint verification of a completed M1 run can proceed without the meshes.

In [ ]:
meshes = repo / "results/m3_robot_assets"
expected_meshes = set("""head_link.STL
left_ankle_pitch_link.STL
left_ankle_roll_link.STL
left_elbow_link.STL
left_hip_pitch_link.STL
left_hip_roll_link.STL
left_hip_yaw_link.STL
left_knee_link.STL
left_rubber_hand.STL
left_shoulder_pitch_link.STL
left_shoulder_roll_link.STL
left_shoulder_yaw_link.STL
left_wrist_pitch_link.STL
left_wrist_roll_link.STL
left_wrist_yaw_link.STL
logo_link.STL
pelvis.STL
pelvis_contour_link.STL
right_ankle_pitch_link.STL
right_ankle_roll_link.STL
right_elbow_link.STL
right_hip_pitch_link.STL
right_hip_roll_link.STL
right_hip_yaw_link.STL
right_knee_link.STL
right_rubber_hand.STL
right_shoulder_pitch_link.STL
right_shoulder_roll_link.STL
right_shoulder_yaw_link.STL
right_wrist_pitch_link.STL
right_wrist_roll_link.STL
right_wrist_yaw_link.STL
torso_link_rev_1_0.STL
waist_roll_link_rev_1_0.STL
waist_yaw_link_rev_1_0.STL""".split())
missing = expected_meshes.difference(p.name for p in meshes.glob("*.STL") if p.stat().st_size)
if missing and not (run / "training.json").is_file():
    meshes.mkdir(parents=True, exist_ok=True)
    source_dir = inputs / "datasets/trung1311/robot-assets/results/m3_robot_assets"
    archives = [] if source_dir.is_dir() else sorted(inputs.rglob("m3-robot-assets.zip"))
    if archives:
        with ZipFile(archives[0]) as bundle:
            for name in sorted(missing):
                with bundle.open(f"results/m3_robot_assets/{name}") as source:
                    (meshes / name).write_bytes(source.read())
    else:
        if not source_dir.is_dir():
            folders = [p for p in inputs.rglob("m3_robot_assets") if p.is_dir()
                       and all((p / name).is_file() for name in missing)]
            if not folders:
                raise FileNotFoundError(f"Attach the robot-assets dataset at {source_dir} or attach m3-robot-assets.zip.")
            source_dir = folders[0]
        for name in sorted(missing):
            shutil.copyfile(source_dir / name, meshes / name)
    missing = {name for name in expected_meshes if not (meshes / name).stat().st_size}
    if missing:
        raise ValueError(f"Empty G1 mesh files: {sorted(missing)}")

## Enable M1 in the checkout

The update targets the training entry point, observation builder, and PPO trainer. A conflicting edit stops the update before training starts.

In [ ]:
patch = r'''diff --git a/scripts/train_kaggle.py b/scripts/train_kaggle.py
index 17d1537..7619bb2 100644
--- a/scripts/train_kaggle.py
+++ b/scripts/train_kaggle.py
@@ -31,33 +31,40 @@ def main():
                         help="Path to terrain configuration YAML")
     parser.add_argument("--num-envs", type=int, default=None,
                         help="Total parallel environments (sharded across GPUs)")
-    parser.add_argument("--history-len", type=int, default=5,
-                        help="Number of history timesteps (k)")
+    parser.add_argument("--history-len", type=int,
+                        help="Number of history timesteps (M1: 0, other models default to 5)")
     parser.add_argument("--seed", type=int, default=42,
                         help="Master PRNG seed")
     parser.add_argument("--output-dir", type=str, default=None,
                         help="Checkpoint and logs output directory")
-    parser.add_argument("--num-timesteps", type=int, help="Exact M3 environment step budget")
-    parser.add_argument("--smoke", action="store_true", help="Run two M3 optimizer updates (pilot default: ten)")
+    parser.add_argument("--num-timesteps", type=int, help="Exact M1 or M3 environment step budget")
+    parser.add_argument("--smoke", action="store_true", help="Run two PPO optimizer updates (pilot default: ten)")
     args = parser.parse_args()
-    if args.model == "M3":
+    if args.history_len is None:
+        args.history_len = 0 if args.model == "M1" else 5
+    if args.model in ("M1", "M3"):
+        if args.model == "M1" and args.history_len != 0:
+            parser.error("M1 uses only the current observation (--history-len 0)")
+        if args.model == "M3" and args.history_len < 1:
+            parser.error("M3 history length must be positive")
         if args.category is not None or args.difficulty != 0 or Path(args.terrain_config).resolve() != ROOT / "configs/terrain.yaml":
-            parser.error("M3 uses the established terrain configuration and curriculum")
-        if args.history_len < 1 or any(v is not None and v <= 0 for v in (args.num_envs, args.num_timesteps)):
-            parser.error("History length, environment count, and step budget must be positive")
+            parser.error(f"{args.model} uses the established terrain configuration and curriculum")
+        if any(v is not None and v <= 0 for v in (args.num_envs, args.num_timesteps)):
+            parser.error("Environment count and step budget must be positive")
         if args.smoke and args.num_timesteps is not None:
             parser.error("Choose either --smoke or an explicit --num-timesteps")
-        args.output_dir = args.output_dir or ("results/m3_smoke" if args.smoke else "results/m3_pilot")
+        kind = "smoke" if args.smoke else "pilot"
+        args.output_dir = args.output_dir or f"results/{args.model.lower()}_{kind}"
         try:
             from src.training.m3_pilot import run
         except ImportError as exc:
-            parser.exit(2, f"M3 needs the configured Playground/Brax runtime: {exc}\n")
+            parser.exit(2, f"{args.model} needs the configured Playground/Brax runtime: {exc}\n")
         run(args)
         # Release the trainer's GPU allocations before verifying in a fresh interpreter.
         os.execv(sys.executable, [sys.executable, "-m", "src.training.m3_pilot", args.output_dir])
 
     if args.num_timesteps is not None or args.smoke:
-        parser.error("--num-timesteps and --smoke apply to M3")
+        parser.error("--num-timesteps and --smoke apply to M1 and M3")
     os.environ.update(XLA_PYTHON_CLIENT_PREALLOCATE="false",
                       XLA_PYTHON_CLIENT_MEM_FRACTION="0.85", TF_FORCE_GPU_ALLOW_GROWTH="true")
     args.num_envs = args.num_envs or 4096
diff --git a/src/training/m3_history.py b/src/training/m3_history.py
index 03ad757..5120aab 100644
--- a/src/training/m3_history.py
+++ b/src/training/m3_history.py
@@ -2,8 +2,10 @@
 
 
 def initial_input(obs, action_width, length, xp):
-    if length < 1:
-        raise ValueError("history length must be positive")
+    if length == 0:
+        return obs
+    if length < 0:
+        raise ValueError("history length must be nonnegative")
     shape = (*obs.shape[:-1], length * (obs.shape[-1] + action_width))
     return xp.concatenate((obs, xp.zeros(shape, dtype=obs.dtype)), axis=-1)
 
@@ -11,6 +13,8 @@ def initial_input(obs, action_width, length, xp):
 def next_input(previous, action, observation, done, xp):
     """Advance one transition. On autoreset, observation is the new reset observation."""
     width = observation.shape[-1]
+    if previous.shape[-1] == width:
+        return observation
     pair_width = width + action.shape[-1]
     history = xp.concatenate(
         (previous[..., width + pair_width:], previous[..., :width], action), axis=-1
diff --git a/src/training/m3_pilot.py b/src/training/m3_pilot.py
index 256e9f4..0881c33 100644
--- a/src/training/m3_pilot.py
+++ b/src/training/m3_pilot.py
@@ -1,4 +1,4 @@
-"""Bounded M3 PPO training with matched initial/final evaluations."""
+"""Bounded M1 and M3 PPO training with matched initial/final evaluations."""
 
 from functools import partial
 from hashlib import sha256
@@ -65,7 +65,7 @@ def evaluate(env, policy, horizon):
             values = np.asarray([state.reward, state.metrics["m3_distance"],
                                  state.metrics["m3_tracking"], state.done])
             if not np.isfinite(values).all():
-                raise ValueError("Non-finite M3 evaluation transition")
+                raise ValueError("Non-finite evaluation transition")
             reward += float(values[0])
             distance += float(values[1])
             error += float(values[2])
@@ -177,7 +177,7 @@ def run(args):
     width = actor_shape.shape[-1]
     expected = width + args.history_len * (width + env.action_size)
     if observation_shapes["state"] != (expected,):
-        raise ValueError("Simulator observation shape disagrees with the M3 history contract")
+        raise ValueError("Simulator observation shape disagrees with the configured actor width")
 
     ppo = locomotion_params.brax_ppo_config(TASK).to_dict()
     ppo["learning_rate"] = 0.0001
@@ -189,7 +189,8 @@ def run(args):
     ppo.update(num_envs=nenv, batch_size=nenv, num_minibatches=1, num_updates_per_batch=1,
                num_timesteps=budget, num_evals=1, num_eval_envs=nenv, num_resets_per_eval=0,
                run_evals=False)
-    settings = dict(model="M3", architecture="PPO MLP with observation–action history stacking",
+    settings = dict(model=args.model, architecture=("PPO MLP with current observation" if args.model == "M1"
+                                                 else "PPO MLP with observation–action history stacking"),
         task=TASK, seed=args.seed, actor_fields=[
         "pelvis gyro", "projected gravity", "velocity command", "joint offsets", "joint velocities"],
         observation_width=width, actions=env.action_size, actor_width=expected,
@@ -201,7 +202,7 @@ def run(args):
         observation_shapes=observation_shapes, optimizer_updates=budget // unit,
         terrain_config=str(terrain_path.resolve()), output_dir=str(out),
         devices=[f"{d}: {d.device_kind}" for d in jax.local_devices()])
-    print("Resolved M3 run configuration", flush=True)
+    print(f"Resolved {args.model} run configuration", flush=True)
     print(json.dumps(settings, indent=2, allow_nan=False), flush=True)
     save_json(out / "settings.json", settings)
 
'''
def check_patch(*flags):
    return subprocess.run(["git", "apply", *flags, "-"], cwd=repo, input=patch,
                          text=True, capture_output=True)

if check_patch("--reverse", "--check").returncode != 0:
    result = check_patch("--check")
    if result.returncode:
        raise RuntimeError("Use the updated m3-kaggle checkout. " + result.stderr)
    check_patch().check_returncode()
print("M1 trainer ready.")

## Select the training Python

Use the Python environment that ran M3. If it lives in a custom location, set `m3_python` below to its Python executable.

In [ ]:
m3_python = None  # Set to the full M3 Python path if known.
if m3_python is not None:
    candidates = [m3_python]
else:
    candidates = [sys.executable]
    for variable in ("VIRTUAL_ENV", "CONDA_PREFIX"):
        if env.get(variable):
            candidates.append(Path(env[variable]) / "bin/python")
    candidates.extend([repo / ".venv/bin/python", shutil.which("python"), shutil.which("python3"),
                       "/opt/conda/bin/python", "/usr/local/bin/python3"])

failures = []
for python in dict.fromkeys(str(path) for path in candidates if path):
    if not os.access(python, os.X_OK) or not Path(python).is_file():
        failures.append(f"{python}: executable unavailable")
        continue
    probe = subprocess.run([python, "-c", "from src.training.m3_pilot import run"],
                           cwd=repo, env=env, text=True, capture_output=True)
    if probe.returncode == 0:
        training_python = python
        print(f"M1 training Python: {training_python}")
        break
    failures.append(f"{python}: {probe.stderr.strip() or probe.stdout.strip() or probe.returncode}")
else:
    raise RuntimeError("Could not import the M1 trainer. Set m3_python to the Python executable "
                       "from the configured M3 environment and rerun this cell.\n" + "\n".join(failures))

## Train and verify the baseline

A completed run goes directly to checkpoint verification. A partial run needs a new output folder for a fresh start.

In [ ]:
def check_m1_run():
    settings = read(run, "settings.json")
    ppo = settings["ppo"]
    expected = ("M1", 0, seed, num_envs, budget, 0.0001)
    actual = (settings["model"], settings["history_length"], settings["seed"],
              ppo["num_envs"], ppo["num_timesteps"], ppo["learning_rate"])
    if actual != expected or read(run, "training.json")["environment_steps"] != budget:
        raise ValueError(f"{run} does not contain the requested completed M1 run.")
    if settings["actor_width"] != settings["observation_width"] or settings["network"]["policy_obs_key"] != "state":
        raise ValueError("Expected an M1 actor using only the current state.")
    return settings

if (run / "training.json").is_file():
    check_m1_run()
    command = [training_python, "-u", "-m", "src.training.m3_pilot", str(run)]
else:
    if run.exists() and any(run.iterdir()):
        raise FileExistsError(f"Partial run at {run}. Choose an empty output directory.")
    probe = """import json
import jax
print(json.dumps([(d.platform, d.device_kind) for d in jax.local_devices()]))
"""
    devices = json.loads(subprocess.check_output([training_python, "-c", probe], cwd=repo, env=env, text=True))
    if len(devices) != 2 or any(platform not in ("gpu", "cuda") or "T4" not in kind.split() for platform, kind in devices):
        raise RuntimeError(f"M1 training requires Kaggle GPU T4 x2. JAX reported {devices}.")
    print(f"JAX sees two T4 GPUs. Training with {num_envs // 2} environments per GPU.", flush=True)
    command = [training_python, "-u", "scripts/train_kaggle.py", "--model", "M1",
               "--history-len", "0", "--num-envs", str(num_envs), "--num-timesteps", str(budget),
               "--seed", str(seed), "--output-dir", str(run)]
subprocess.run(command, cwd=repo, env=env, check=True)
m1 = check_m1_run()
reload_result = read(run, "reload.json")
if reload_result["action_verified"] is not True or int(reload_result["checkpoint"]) != budget:
    raise ValueError(f"Expected a verified M1 checkpoint at step {budget}.")
print(f"Verified M1 checkpoint at {budget:,} steps in {run}")

## Evaluate M1

Compare episode duration, planar velocity tracking error, return and early terminations on the same ten seeds. An early termination can include a torso inversion or a leg collision.

In [ ]:
def report(folder, settings, expected_seeds=None):
    print("Policy / stage       Duration    Tracking error    Return    Early ends")
    for stage in ("initial", "trained"):
        episodes = read(folder, f"{stage}_evaluation.json")["episodes"]
        seeds = sorted(e["seed"] for e in episodes)
        if len(seeds) != 10 or len(set(seeds)) != 10:
            raise ValueError(f"Expected ten distinct {stage} evaluation seeds in {folder}.")
        if expected_seeds is not None and seeds != expected_seeds:
            raise ValueError(f"Unmatched {stage} evaluation seeds in {folder}.")
        expected_seeds = seeds
        duration = statistics.mean(e["steps"] for e in episodes) * settings["control_dt"]
        error = statistics.mean(e["tracking_error_mps"] for e in episodes)
        reward = statistics.mean(e["episode_return"] for e in episodes)
        ended = sum(e["fall"] for e in episodes)
        print(f'{settings["model"]} / {stage:<8}      {duration:.3f} s       {error:.3f} m/s'
              f'     {reward:7.3f}      {ended}/10')
    return expected_seeds

m1_seeds = report(run, m1)

## Optional comparison with M3

Attach a saved M3 run and rerun the next cell when you are ready. Set `reference` to the run directory if it has a different name or there are multiple matches.

In [ ]:
reference = None  # Optionally set to Path("/kaggle/input/.../your-m3-run").
if reference is None:
    name = "m3_1p6m_env8_lr1e4"
    local = repo / "results" / name
    if (local / "settings.json").is_file():
        reference = local
    else:
        matches = sorted(inputs.rglob(f"{name}/settings.json"))
        if len(matches) == 1:
            reference = matches[0].parent
        elif len(matches) > 1:
            print("Multiple M3 runs found. Set reference to one of:", [p.parent for p in matches])

if reference is None:
    print("M1 evaluation complete. M3 comparison skipped.")
else:
    m3 = read(reference, "settings.json")
    if (m3["model"], m3["history_length"]) != ("M3", 5):
        raise ValueError("Select a five-step M3 reference run.")
    if read(reference, "training.json")["environment_steps"] != budget:
        raise ValueError(f"Expected a completed {budget:,}-step M3 run.")
    for key in ("task", "seed", "terrain_sha256", "terrain", "difficulty", "ppo", "network",
                "actor_fields", "observation_width", "actions", "control_dt", "physics_dt",
                "simulator_backend", "warp_contact_capacity"):
        if m1[key] != m3[key]:
            raise ValueError(f"Run settings differ for {key}: {m1[key]} vs {m3[key]}")
    critic_key = m1["network"]["value_obs_key"]
    if m1["observation_shapes"][critic_key] != m3["observation_shapes"][critic_key]:
        raise ValueError("Critic input shapes differ.")
    if m1["packages"] != m3["packages"]:
        print("Runtime versions differed between runs. M1:", m1["packages"], "M3:", m3["packages"])
    report(reference, m3, m1_seeds)